In [1]:
import os
import json

from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

from dotenv import load_dotenv

load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One Provider described as pure DATA."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS=[
    Provider("Groq", "GROQ_API_KEY", True, None, "openai/gpt-oss-20b"),
]


def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected=", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file")


def build_client(provider: Provider):
    from groq import Groq

    api_key=os.environ[provider.env_var]
    if provider.base_url is None:
        return Groq(api_key=api_key)
    return Groq(api_key=api_key, base_url=provider.base_url)


def llm_reply(prompt: str,*,max_tokens: int=300)->str:
    """Send one user prompt; return the assistant's text."""

    provider=select_provider()
    client=build_client(provider)

    result=client.chat.completions.create(
        model=provider.model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return result.choices[0].message.content

In [2]:
WEATHER_DB={
    "London": {"celcius": 22, "sky": "cloudy"},
    "Paris": {"celcius": 24, "sky": "sunny"},
    "Moscow": {"celcius": 28, "sky": "synny"},
    "Tokyo": {"celcius": 30, "sky": "cloudy"},
    "Bengaluru": {"celcius": 25, "sky": "rainy"},
    "New York": {"celcius": 29, "sky": "cloudy"},
    "Cape Town": {"celcius": 34, "sky": "rainy"},
    "Sydney": {"celcius": 33, "sky": "sunny"},
}

In [3]:
def lookup_weather(location: str)->str:
    """Lookup to the weather for a location"""
    record=WEATHER_DB.get(location)
    if record is None:
        return f"Sorry, I dont know the weather in {location}."
    return f"The weather in {location} is {record['celcius']}C and {record['sky']}."

In [4]:
#This schema will be given to the LLM
weather_schema={
    "type": "function",
    "function": {
        "name": "lookup_weather",
        "description": "Look up the weather for a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The location to look up the weather for."}
            },
            "required": ["location"]
        }
    }
}

TOOLS={
    "lookup_weather": lookup_weather
}

In [5]:
def ask_llm_with_tool(prompt: str,*,max_tokens: int=300)->str:
    """Call the LLM with a tool call"""

    provider=select_provider()
    client=build_client(provider)

    result=client.chat.completions.create(
        model=provider.model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=max_tokens,
        tools=[weather_schema],
    )
    print(result.choices)
    return result.choices[0].message

In [6]:
mssg=ask_llm_with_tool("What is the weather in Tokyo?")
print("final response from llm",mssg)

[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to use function call lookup_weather with location Tokyo.', tool_calls=[ChatCompletionMessageToolCall(id='fc_072e6bd2-214f-4ef7-bbc8-78c46c9bd999', function=Function(arguments='{"location":"Tokyo"}', name='lookup_weather'), type='function')]))]
final response from llm ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to use function call lookup_weather with location Tokyo.', tool_calls=[ChatCompletionMessageToolCall(id='fc_072e6bd2-214f-4ef7-bbc8-78c46c9bd999', function=Function(arguments='{"location":"Tokyo"}', name='lookup_weather'), type='function')])


In [ ]:
"""
    Output from LLm was:
[
    {
        "finish_reason": "tool_calls",
        "index": 0,
        "logprobs": null,
        "message": {
            "annotations": null,
            "content": null,
            "executed_tools": null,
            "function_call": null,
            "reasoning": "We need to use the function lookup_weather with location Tokyo.",
            "role": "assistant",
            "tool_calls": [
                {
                "function": {
                    "arguments": "{\"location\":\"Tokyo\"}",
                    "name": "lookup_weather"
                },
                "id": "fc_072e6bd2-214f-4ef7-bbc8-78c46c9bd999",
                "type": "function"
                }
            ]
        }
    }
]
"""

if mssg.tool_calls:
    print("Tool Call detected")
    print(f"{len(mssg.tool_calls)} Tool Calls required")
    for tool_call in mssg.tool_calls:
        print("Tool Call name",tool_call.function.name)
        print("Tool Call arguments",tool_call.function.arguments)
        city=json.loads(tool_call.function.arguments)["location"]
        tool=TOOLS[tool_call.function.name] # This will give us the function
        result=tool(city)
        print(result)
else:
    print("No Tool Calls detected")

Tool Call detected
1 Tool Calls required
Tool Call name lookup_weather
Tool Call arguments {"location":"Tokyo"}
The weather in Tokyo is 30C and cloudy.


In [ ]:
def ask_llm_with_tool(prompt: str,*,max_tokens: int=300)->str:
    """Call the LLM with a tool call"""

    provider=select_provider()
    client=build_client(provider)
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]

    while True:
        response=client.chat.completions.create(
            model=provider.model,
            messages=messages,
            max_tokens=max_tokens,
            tools=[weather_schema],
        )
        msg=response.choices[0].message

        if not msg.tool_calls:
            # llm is done and we can return the response
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": msg.tool_calls
        })

        # llm is asking us to call a tool
        for tool_call in msg.tool_calls:
            tool_name=tool_call.function.name
            tool_args=json.loads(tool_call.function.arguments)
            tool_call_id=tool_call.id
            
            if tool_name not in TOOLS:
                raise ValueError(f"{tool_name} not found")

            tool=TOOLS[tool_name] # actual function
            result=tool(**tool_args) # getting response of the tool calls

            # appending the response back in the messages context array
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "content": result
                }
            )

In [13]:
ask_llm_with_tool("Whats the weather like in London compared to Paris?")

'London is a bit cooler and overcast – about **22\u202f°C and cloudy** – while Paris is warmer and clearer, sitting at **24\u202f°C and sunny**. So Paris is roughly 2\u202f°C warmer and has a sunnier outlook than London.'